# 04 · Topic Modeling — BERTopic on Captions & Justifications

**Pipeline**: `sentence-transformers/all-MiniLM-L6-v2` → UMAP → HDBSCAN → c-TF-IDF (BERTopic)

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

from src.config import FIGURES, OUTPUTS, FONT_SCALE, SENT_COLORS, SENT_COLORS_3
from src.data_loading import load_annotations, parse_demographics, create_profiles
from src.embeddings import load_or_encode_with_ids
from src.topic_modeling import fit_bertopic, topic_sentiment_composition, save_topic_assignments

MUTED = sns.color_palette("muted")
sns.set_theme(style="whitegrid", font_scale=FONT_SCALE)
plt.rcParams["figure.dpi"] = 150

FIGURES.mkdir(exist_ok=True)
OUTPUTS.mkdir(exist_ok=True)

df = load_annotations()
df = parse_demographics(df)
df = create_profiles(df)
print(f"Records: {len(df):,} | Images: {df['image_id'].nunique():,} | Personas: {df['persona_id'].nunique():,}")

## 1 · Encode texts (reuse or compute embeddings)

In [ ]:
CAPTION_EMBED_CACHE = OUTPUTS / "caption_embeddings.npy"
JUST_EMBED_CACHE    = OUTPUTS / "justification_embeddings.npy"
ID_CACHE            = OUTPUTS / "caption_embeddings_ids.csv"

cap_embs,  cap_idx  = load_or_encode_with_ids(df, "caption",       CAPTION_EMBED_CACHE, ID_CACHE)
just_embs, just_idx = load_or_encode_with_ids(df, "justification", JUST_EMBED_CACHE,    ID_CACHE)
print(f"Caption embeddings:       {cap_embs.shape}")
print(f"Justification embeddings: {just_embs.shape}")

## 2 · Fit BERTopic on captions

In [ ]:
CAPTION_TOPIC_CACHE = OUTPUTS / "bertopic_captions.pkl"

cap_texts = df["caption"].tolist()
topic_model_captions, cap_topics, cap_probs = fit_bertopic(cap_texts, cap_embs)
df["caption_topic"] = cap_topics
print(f"Caption topics: {len(set(cap_topics)) - 1} topics + noise")

## 3 · Top words per caption topic

## 4 · Fit BERTopic on justifications

In [ ]:
just_texts = df["justification"].tolist()
topic_model_just, just_topics, just_probs = fit_bertopic(just_texts, just_embs)
df["just_topic"] = just_topics
print(f"Justification topics: {len(set(just_topics)) - 1} topics + noise")

## 5 · Topic distribution by predicted sentiment (justifications)

## 4.5 · Top words per justification topic

In [ ]:
import numpy as np
from matplotlib.ticker import FuncFormatter

# ── Short descriptive labels for the top 10 justification topics ───────────
_TOPIC_LABELS = {
    0: "Natural landscape & beauty",
    1: "Work & development",
    2: "Accident & police response",
    3: "Rural & quiet road",
    4: "Historical & preserved",
    5: "Destruction & conflict",
    6: "Streets & orderly urban",
    7: "City & nature",
    8: "Home & peaceful scene",
    9: "Typical urban scene",
}

just_info = topic_model_just.get_topic_info()
top_just_topics = just_info[just_info["Topic"] != -1].head(10)["Topic"].tolist()

sent_topic = (
    df[
        df["just_topic"].isin(top_just_topics)
        & df["predicted_sentiment"].isin(["Positive", "Neutral", "Negative"])
    ]
    .groupby(["just_topic", "predicted_sentiment"])
    .size()
    .reset_index(name="count")
)

pivot = sent_topic.pivot(
    index="just_topic", columns="predicted_sentiment", values="count"
).fillna(0)
pivot_norm = pivot.div(pivot.sum(axis=1), axis=0)[["Positive", "Neutral", "Negative"]]
pivot_norm.index = [_TOPIC_LABELS.get(i, str(i)) for i in pivot_norm.index]

# Sort ascending by Negative → least negative at bottom, most negative at top
pivot_norm = pivot_norm.sort_values("Negative", ascending=True)

# ── Plot ───────────────────────────────────────────────────────────────────
COLS   = ["Positive", "Neutral", "Negative"]
COLORS = [SENT_COLORS_3[c] for c in COLS]

fig, ax = plt.subplots(figsize=(9, 5.5))
left = np.zeros(len(pivot_norm))

for col, color in zip(COLS, COLORS):
    vals = pivot_norm[col].values
    ax.barh(range(len(pivot_norm)), vals, left=left,
            color=color, label=col, height=0.62,
            edgecolor="white", linewidth=0.5)
    for i, (val, l) in enumerate(zip(vals, left)):
        if val > 0.09:
            ax.text(l + val / 2, i, f"{val:.0%}",
                    ha="center", va="center", fontsize=9,
                    color="white", fontweight="bold")
    left += vals

ax.set_yticks(range(len(pivot_norm)))
ax.set_yticklabels(pivot_norm.index, fontsize=11)
ax.set_xlabel("Proportion", fontsize=11)
ax.set_xlim(0, 1.0)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.tick_params(axis="x", labelsize=10)
ax.xaxis.grid(True, linestyle="--", alpha=0.35, color="gray", zorder=0)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.legend(
    title="Sentiment", bbox_to_anchor=(1.01, 0.5), loc="center left",
    fontsize=10, title_fontsize=10, framealpha=0.9,
)

plt.tight_layout()
fig.savefig(FIGURES / "fig_topic_just_sentiment_composition.pdf", bbox_inches="tight")
fig.savefig(FIGURES / "fig_topic_just_sentiment_composition.png", bbox_inches="tight")
plt.show()


## 6 · Save topic assignments

In [ ]:
df[["annotation_id", "persona_id", "image_id",
    "predicted_sentiment", "caption_topic", "just_topic"]].to_csv(
    OUTPUTS / "annotations_with_topics.csv", index=False
)

topic_model_captions.get_topic_info().to_csv(OUTPUTS / "caption_topic_info.csv", index=False)
topic_model_just.get_topic_info().to_csv(OUTPUTS / "just_topic_info.csv", index=False)
print("Saved topic assignments and topic info.")

## 7 · Topic × Persona profile heatmaps



In [ ]:
# Load topic assignments + join demographics
_tdf = pd.read_csv(OUTPUTS / "annotations_with_topics.csv")
_mdf = _tdf.merge(df[["annotation_id", "profile_abbr"]], on="annotation_id", how="left")

_labeled_topics = list(_TOPIC_LABELS.keys())
_tp = (_mdf[_mdf["just_topic"].isin(_labeled_topics)]
       .groupby(["just_topic", "profile_abbr"]).size().reset_index(name="count"))
_piv   = _tp.pivot(index="just_topic", columns="profile_abbr", values="count").fillna(0)
_piv_n = _piv.div(_piv.sum(axis=0), axis=1)
_piv_n.index = [_TOPIC_LABELS[k] for k in _piv_n.index]

_tp_pivots = {"just": _piv_n}
print(f"just pivot shape: {_piv_n.shape}")


In [ ]:
sns.set_theme(style="whitegrid", font_scale=FONT_SCALE)

_TOPIC_LABELS = {
    0: "Natural landscape & beauty",
    1: "Work & development",
    2: "Accident & police response",
    3: "Rural & quiet road",
    4: "Historical & preserved",
    5: "Destruction & conflict",
    6: "Streets & orderly urban",
    7: "City & nature",
    8: "Home & peaceful scene",
    9: "Typical urban scene",
}

def _tp_heatmap(pivot, fname, topic_labels=None, annot=False):
    display_pivot = pivot.copy()
    vmin = display_pivot.values.min()
    vmax = display_pivot.values.max()
    fig, ax = plt.subplots(figsize=(18, 7))
    sns.heatmap(
        display_pivot, ax=ax, cmap="YlOrRd",
        vmin=vmin, vmax=vmax,
        annot=annot, fmt=".2f", annot_kws={"fontsize": 8},
        linewidths=0.3, linecolor="white",
        cbar_kws={"label": "Topic proportion", "shrink": 0.6},
    )
    ax.set_xlabel("Persona", fontsize=15)
    ax.set_ylabel("Topic", fontsize=15)
    ax.tick_params(axis="x", rotation=90, labelsize=10)
    ax.tick_params(axis="y", rotation=0, labelsize=11)
    plt.tight_layout()
    fig.savefig(FIGURES / f"{fname}.pdf", bbox_inches="tight")
    fig.savefig(FIGURES / f"{fname}.png", bbox_inches="tight")
    plt.show()
    print(f"Saved {fname}")

_tp_heatmap(_tp_pivots["just"],    "fig_topic_persona_heatmap_just", topic_labels=_TOPIC_LABELS, annot=True)


In [ ]:
## 8 · t-SNE and PCA of persona profiles in topic-distribution space
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.patches as mpatches

sns.set_theme(style="whitegrid", font_scale=FONT_SCALE)

piv_just = _tp_pivots["just"].T  # shape: (n_profiles, n_topics)
profiles  = piv_just.index.tolist()

X = StandardScaler().fit_transform(piv_just.values)
emb_tsne = TSNE(n_components=2, perplexity=5, random_state=42, max_iter=2000).fit_transform(X)
emb_pca  = PCA(n_components=2, random_state=42).fit_transform(X)

def _parse_profile(p):
    parts = [x.strip() for x in p.split("/")]
    econ  = "High" if any("High" in x or "high" in x for x in parts) else "Low"
    polit = "Con" if any("Con" in x or "con" in x.lower() for x in parts) else "Pro"
    return econ, polit

attrs      = [_parse_profile(p) for p in profiles]
econ_vals  = [a[0] for a in attrs]
polit_vals = [a[1] for a in attrs]

econ_colors  = {"High": "#2166ac", "Low": "#d73027"}
polit_shapes = {"Con": "^", "Pro": "o"}

legend_handles = [
    mpatches.Patch(color=econ_colors["High"], label="High income"),
    mpatches.Patch(color=econ_colors["Low"],  label="Low income"),
    plt.Line2D([0],[0], marker="^", color="gray", markersize=11, linestyle="", label="Conservative"),
    plt.Line2D([0],[0], marker="o", color="gray", markersize=11, linestyle="", label="Progressive"),
]

# t-SNE only, no title
fig_tsne, ax_tsne = plt.subplots(figsize=(9, 7))
for i, (prof, econ, polit) in enumerate(zip(profiles, econ_vals, polit_vals)):
    ax_tsne.scatter(emb_tsne[i, 0], emb_tsne[i, 1],
                    c=econ_colors[econ], marker=polit_shapes[polit],
                    s=350, edgecolors="white", linewidths=0.7, zorder=3)
ax_tsne.legend(handles=legend_handles, fontsize=13, loc="best")
ax_tsne.set_xlabel("t-SNE dim 1", fontsize=15)
ax_tsne.set_ylabel("t-SNE dim 2", fontsize=15)
ax_tsne.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
fig_tsne.savefig(FIGURES / "fig_persona_topic_tsne.pdf", bbox_inches="tight")
fig_tsne.savefig(FIGURES / "fig_persona_topic_tsne.png", bbox_inches="tight")
plt.show()
print("Saved fig_persona_topic_tsne")

# PCA only, no title
fig_pca, ax_pca = plt.subplots(figsize=(9, 7))
for i, (prof, econ, polit) in enumerate(zip(profiles, econ_vals, polit_vals)):
    ax_pca.scatter(emb_pca[i, 0], emb_pca[i, 1],
                   c=econ_colors[econ], marker=polit_shapes[polit],
                   s=350, edgecolors="white", linewidths=0.7, zorder=3)
ax_pca.legend(handles=legend_handles, fontsize=13, loc="best")
ax_pca.set_xlabel("PC 1", fontsize=15)
ax_pca.set_ylabel("PC 2", fontsize=15)
ax_pca.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
fig_pca.savefig(FIGURES / "fig_persona_topic_pca.pdf", bbox_inches="tight")
fig_pca.savefig(FIGURES / "fig_persona_topic_pca.png", bbox_inches="tight")
plt.show()
print("Saved fig_persona_topic_pca")
